# Week 5 · Day 4 — CrewAI
### Multi-Agent Collaboration, Roles & Task Delegation

Day 3 built one generalist agent (`StateGraph`) that planned, retrieved,
generated, critiqued, and revised its own laptop-purchase recommendation —
all inside a single reasoning loop. Today's notebook takes the **same
domain** (the laptop catalog from Day 2/3) and asks: what if that one job
were split across a small team of specialists instead?

**CrewAI**'s framing is a *crew* of `Agent`s, each with a `role`, `goal`, and
`backstory`, wired to `Task`s and (optionally) `tools`, run through a
`Process` — `sequential` (a fixed pipeline, closest to Day 3's linear
`plan → retrieve → generate` path) or `hierarchical` (a manager agent
delegates and reviews). We build both, compare them to each other and to
Day 3's single-agent graph, and log token usage throughout.

> **Environment note from Umer:** this notebook is written to run end-to-end against
 my Gemini API. So if you want to run this notebook make sure to add your Gemini API key.

## Step 0 — Install & load the Gemini API key

Same key and same portable Colab-or-local loader pattern as Day 2/3.
CrewAI talks to Gemini through `litellm` under the hood, so the model
string needs a `gemini/` provider prefix (`gemini/gemini-3.5-flash-lite`)
instead of the bare name LangChain used.

In [1]:
%pip install -q -U crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.3/195.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [2]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### A note on running crews inside a notebook

Jupyter/Colab kernels run their own asyncio event loop for the whole
session. This version of CrewAI explicitly checks for that
(`asyncio.get_running_loop()`) inside its synchronous execution path and
refuses to proceed if one is already running, rather than trying to nest
loops — so `nest_asyncio`-style patching doesn't help here, since CrewAI
isn't calling `run_until_complete()` in that branch; it's just checking
"is a loop running?" and raising immediately if so. The fix the error
message itself points to is the right one: call **`await
crew.kickoff_async()`** instead of `crew.kickoff()`. Top-level `await`
works fine directly in a Jupyter/Colab cell, so no extra plumbing is
needed -- every `kickoff()` call below uses the async form for this
reason.

In [3]:
import os

def load_gemini_api_key() -> str:
    # Prefer Colab's secret manager when running in Colab; fall back to a
    # plain environment variable everywhere else (local Jupyter, CI, etc.).
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    raise ValueError(
        "GEMINI_API_KEY not found. In Colab: add it via the key icon in the "
        "left sidebar (name it GEMINI_API_KEY, enable notebook access). "
        "Elsewhere: `export GEMINI_API_KEY=...` before starting Jupyter, or "
        "`os.environ['GEMINI_API_KEY'] = '...'` in a cell above this one."
    )

GEMINI_API_KEY = load_gemini_api_key()
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY  # litellm reads this env var directly
print("Gemini API key loaded.")

Gemini API key loaded.


In [4]:
from crewai import Agent, Task, Crew, Process, LLM

MODEL_NAME = "gemini/gemini-3.5-flash-lite"  # litellm provider-prefixed form
llm = LLM(model=MODEL_NAME, api_key=GEMINI_API_KEY, temperature=0.3)
manager_llm = LLM(model=MODEL_NAME, api_key=GEMINI_API_KEY, temperature=0.2)

# The Gemini free tier caps gemini-3.5-flash-lite at 15 requests/minute.
# Hierarchical delegation adds manager-level calls on top of the 3 worker
# calls, so it can burn through that quota faster than sequential does --
# keep every agent (and each crew as a whole) comfortably under the limit.
AGENT_MAX_RPM = 8

print("Gemini LLM ready for CrewAI:", MODEL_NAME)

Gemini LLM ready for CrewAI: gemini/gemini-3.5-flash-lite


## Task 1 — Multi-agent design thinking

### The task

**"Given a client's laptop shortlist, research the candidate products,
analyze the pricing/spec trade-offs, and write a stakeholder-ready
purchase recommendation."**

This is deliberately the *same underlying job* Day 3's single graph did
(`retrieve → generate → critique`), so the comparison in Tasks 4–5 is
apples-to-apples: one team, one job, two different ways of organizing the
work.

### Three roles, no overlap

| Agent | Role | Goal | Backstory |
|---|---|---|---|
| **Product Researcher** | Laptop Product Researcher | Retrieve accurate, current specs and pricing for every candidate laptop named in the request — nothing more, nothing interpreted yet. | A former retail electronics buyer who has spent a decade cataloguing hardware specs; obsessive about not stating a number without a source, and never editorializes about which product is "better." |
| **Pricing & Value Analyst** | Pricing & Value Analyst | Turn the researcher's raw specs into concrete, computed comparisons — price deltas, price-per-GB-RAM, and the trade-offs a buyer actually cares about. | A former financial analyst who moved into consumer tech; thinks in ratios and deltas, distrusts any comparison that isn't backed by an actual calculation. |
| **Recommendation Writer** | Client-Facing Recommendation Writer | Turn the analyst's numbers into a short, clear, non-technical recommendation a budget-conscious client can act on immediately. | A client-facing consultant who has delivered hundreds of purchase recommendations; ruthless about cutting jargon, always closes with one explicit recommendation instead of a wall of options. |

Each role owns a distinct **stage** of the pipeline (gather → compute →
communicate) and a distinct **skill** (retrieval, arithmetic, persuasive
writing) — none of them could silently do another's job without stepping
outside their tools or their framing.

### Why a crew, and where it isn't worth it

Splitting the work helps here because each stage rewards a different kind
of prompting and a different failure mode to guard against: the researcher
should be penalized for inventing a number, the analyst for arithmetic
mistakes, the writer for burying the recommendation in hedging — bundling
all three into one generalist prompt makes it easy for the model to blend
those concerns and, say, "compute" a price delta in prose while also
trying to sound persuasive about it. Specialization also makes each stage
independently inspectable and swappable (a new analyst prompt doesn't
touch the writer). **Where it isn't true:** for a task this small — four
laptops, one arithmetic fact, one paragraph of prose — a single
well-designed agent (Day 3's graph, or even a plain prompt) can do all
three steps correctly in one pass; the crew adds real latency and token
cost (three LLM calls plus inter-agent hand-off overhead instead of one)
for a quality gain that's marginal at this scale. The multi-agent
structure starts paying for itself once the task grows — more products,
messier source data, or a genuine need to parallelize research across
many items at once.

## Task 2 — Build agents & assign tools

### Reused from Day 2/3: `calculator` and `lookup_product_price`

Same catalog, same two plain-Python functions as Day 2/3 — wrapped here
as CrewAI `@tool`-decorated functions so agents can call them.

In [5]:
import json
import ast
import operator as op

PRODUCTS_FILE = "products.json"

catalog = [
    {"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0,  "ram_gb": 8,  "storage_gb": 256,  "rating": 4.3},
    {"name": "UltraBook Pro 14", "category": "laptop", "price_usd": 1499.0, "ram_gb": 16, "storage_gb": 512,  "rating": 4.6},
    {"name": "UltraBook Pro 16", "category": "laptop", "price_usd": 1899.0, "ram_gb": 32, "storage_gb": 1024, "rating": 4.7},
    {"name": "ValueBook 14",     "category": "laptop", "price_usd": 599.0,  "ram_gb": 8,  "storage_gb": 256,  "rating": 4.0},
]
with open(PRODUCTS_FILE, "w") as f:
    json.dump(catalog, f, indent=2)

_SAFE_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Mod: op.mod, ast.Pow: op.pow, ast.USub: op.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Disallowed expression element: {ast.dump(node)}")

def _calculator_impl(expression: str) -> str:
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return json.dumps({"success": True, "expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"success": False, "error": f"Could not evaluate '{expression}': {e}"})

def _lookup_product_price_impl(product_name: str) -> str:
    try:
        with open(PRODUCTS_FILE, "r") as f:
            products = json.load(f)
    except Exception as e:
        return json.dumps({"success": False, "error": f"Could not read catalog: {e}"})
    key = product_name.strip().lower()
    matches = [p for p in products if key in p["name"].lower()]
    if not matches:
        return json.dumps({"success": False, "error": f"No product matching '{product_name}'."})
    return json.dumps({"success": True, "matches": matches})

print(_calculator_impl("1499.0 - 899.0"))
print(_lookup_product_price_impl("Air 13"))

{"success": true, "expression": "1499.0 - 899.0", "result": 600.0}
{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}


In [6]:
from crewai.tools import tool
from crewai_tools import FileReadTool

@tool("Lookup product price")
def lookup_product_price_tool(product_name: str) -> str:
    '''Look up price and specs for a laptop by (partial, case-insensitive)
    name in the product catalog. Returns JSON with the matching product(s).'''
    return _lookup_product_price_impl(product_name)

@tool("Calculator")
def calculator_tool(expression: str) -> str:
    '''Evaluate a basic arithmetic expression, e.g. '1499.0 - 899.0'.
    Returns JSON with the numeric result.'''
    return _calculator_impl(expression)

# Built-in CrewAI tool: lets the researcher skim the whole catalog file
# instead of only ever looking up names it already knows to ask for.
catalog_file_tool = FileReadTool(file_path=PRODUCTS_FILE)

print("Tools ready:", lookup_product_price_tool.name, "|", calculator_tool.name, "|", catalog_file_tool.name)

Tools ready: Lookup product price | Calculator | Read a file's content


### Tool assignment — role-appropriate, not "give everyone everything"

| Agent | Tools | Why these, and no more |
|---|---|---|
| Product Researcher | `lookup_product_price_tool`, `catalog_file_tool` | Its whole job is retrieval — one tool for a targeted lookup by name, one for skimming the raw catalog file when the request is vague about exact names. It gets **no** calculator: computing deltas isn't its job, and giving it math access invites it to sneak analysis into what should be raw data. |
| Pricing & Value Analyst | `calculator_tool` | Its whole job is arithmetic on numbers the researcher already retrieved (passed in via task `context`, not looked up again) — it doesn't need catalog access, it needs to compute correctly on the data it's handed. |
| Recommendation Writer | *(none)* | Pure synthesis of the analyst's already-computed numbers into client-ready prose. Giving it tools would let it go re-fetch or re-compute things instead of faithfully reporting what upstream agents already produced — the one failure mode a writer-stage agent shouldn't be able to cause. |

### The three agents

In [7]:
researcher = Agent(
    role="Laptop Product Researcher",
    goal=(
        "Retrieve accurate, sourced specs and pricing for every laptop named "
        "in the client's request. Report only what the catalog says -- never "
        "estimate, round, or editorialize about which product is better."
    ),
    backstory=(
        "A former retail electronics buyer with a decade of cataloguing "
        "hardware specs. Obsessive about sourcing every number from the "
        "catalog tools rather than memory, because a wrong spec upstream "
        "poisons every recommendation built on top of it."
    ),
    tools=[lookup_product_price_tool, catalog_file_tool],
    llm=llm,
    allow_delegation=False,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

analyst = Agent(
    role="Pricing & Value Analyst",
    goal=(
        "Turn raw specs and prices into concrete, calculated comparisons: "
        "price delta, price-per-GB-RAM, and the concrete trade-offs between "
        "the candidate laptops. Every number must come from the calculator "
        "tool, never mental math."
    ),
    backstory=(
        "A former financial analyst who moved into consumer tech. Thinks in "
        "ratios and deltas, and distrusts any comparison -- including their "
        "own -- that isn't backed by an actual computed number."
    ),
    tools=[calculator_tool],
    llm=llm,
    allow_delegation=False,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

writer = Agent(
    role="Client-Facing Recommendation Writer",
    goal=(
        "Turn the analyst's computed comparison into a short, non-technical, "
        "stakeholder-ready recommendation that ends in one explicit pick, "
        "grounded only in the numbers the analyst already computed."
    ),
    backstory=(
        "A client-facing consultant who has delivered hundreds of purchase "
        "recommendations. Ruthless about cutting jargon and hedging, and "
        "never leaves a client without one clear next step."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

print("Agents ready:", researcher.role, "|", analyst.role, "|", writer.role)

Agents ready: Laptop Product Researcher | Pricing & Value Analyst | Client-Facing Recommendation Writer


## Task 3 — Define tasks & process (sequential)

`context=[...]` is what wires later tasks to earlier tasks' outputs --
CrewAI injects the referenced tasks' `output` text directly into the
later task's prompt, which is the mechanism Task 3 asks us to use for
"task dependencies.

In [8]:
CLIENT_REQUEST = (
    "Compare the UltraBook Air 13 and the UltraBook Pro 14 for a "
    "budget-conscious client who mostly does email, browsing, and light "
    "office work."
)

research_task = Task(
    description=(
        f"The client's request is: '{CLIENT_REQUEST}'\n\n"
        "Use the lookup_product_price tool to retrieve the full spec sheet "
        "for BOTH laptops named in the request. Do not compute anything and "
        "do not recommend anything -- only report what the tool returns."
    ),
    expected_output=(
        "A structured list with exactly one entry per laptop, each entry "
        "using this exact key: value format (one laptop per block, blank "
        "line between blocks):\n"
        "Name: <name>\nPrice: $<price_usd>\nRAM: <ram_gb>GB\n"
        "Storage: <storage_gb>GB\nRating: <rating>\n"
        "No prose, no comparison, no recommendation -- data only."
    ),
    agent=researcher,
)

analysis_task = Task(
    description=(
        "Using ONLY the structured spec data produced by the research task "
        "(do not re-look-up anything), use the calculator tool to compute: "
        "(1) the price delta between the two laptops, (2) price-per-GB-RAM "
        "for each laptop, and (3) a one-line trade-off statement (what the "
        "more expensive laptop gets you per extra dollar)."
    ),
    expected_output=(
        "A structured comparison with these exact labeled fields:\n"
        "Price delta: $<value>\n<Laptop A> price-per-GB-RAM: $<value>\n"
        "<Laptop B> price-per-GB-RAM: $<value>\nTrade-off: <one sentence>\n"
        "Every numeric value must be the literal output of a calculator "
        "tool call, not estimated."
    ),
    agent=analyst,
    context=[research_task],
)

writing_task = Task(
    description=(
        f"The client's original request was: '{CLIENT_REQUEST}'\n\n"
        "Using ONLY the analyst's computed comparison, write a short "
        "client-ready recommendation. Cite the exact price delta and at "
        "least one other computed number. End with exactly one explicit "
        "recommended laptop -- no 'it depends' hedging."
    ),
    expected_output=(
        "A 3-5 sentence recommendation in plain, non-technical language, "
        "citing at least two numbers from the analysis, ending with one "
        "clearly stated recommended product."
    ),
    agent=writer,
    context=[research_task, analysis_task],
)

sequential_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

sequential_result = await sequential_crew.kickoff_async()
print("\n=== FINAL RECOMMENDATION (sequential) ===\n")
print(sequential_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 84cc1534-1af0-4ea6-90e5-60838acc86b6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The client's request is: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a budget-conscious   │
│  client who mostly does email, browsing, and light office work.'                                                │
│                                                                                                                 │
│  Use the lookup_product_price tool to retrieve the full spec sheet for BOTH laptops named in the request. Do    │
│  not compute anything and do not recommend anything -- only report what the tool returns.                       │
│  ID: caf1218a-5c81-4e04-8cf6-5e65581dc2c3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Laptop Product Researcher                                                                               │
│                                                                                                                 │
│  Task: The client's request is: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a budget-conscious   │
│  client who mostly does email, browsing, and light office work.'                                                │
│                                                                                                                 │
│  Use the lookup_product_price tool to retrieve the full spec sheet for BOTH laptops named in the request. Do    │
│  not compute anything and do not recommend anything -- only report what the tool returns.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: lookup_product_price                                                                                     │
│  Args: {'product_name': 'UltraBook Air 13'}                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool lookup_product_price executed with result: {"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}...
Tool lookup_product_price executed with result: {"success": true, "matches": [{"name": "UltraBook Pro 14", "category": "laptop", "price_usd": 1499.0, "ram_gb": 16, "storage_gb": 512, "rating": 4.6}]}...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Client-Facing Recommendation Writer                                                                     │
│                                                                                                                 │
│  Task: The client's original request was: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a          │
│  budget-conscious client who mostly does email, browsing, and light office work.'                               │
│                                                                                                                 │
│  Using ONLY the analyst's computed comparison, write a short client-ready recommendation. Cite the exact price  │
│  delta and at least one other computed number. End with exactly one explicit recommended laptop -- no 'it       │
│  depends' hedging.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Client-Facing Recommendation Writer                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  For your daily routine of email, web browsing, and light office work, paying an extra $600.00 for the          │
│  UltraBook Pro 14 does not make financial sense. The UltraBook Air 13 easily handles these tasks while saving   │
│  you a significant amount of money. Furthermore, its user rating of 4.3 out of 5 proves it is a reliable and    │
│  well-liked machine for everyday use.                                                                           │
│                                                                                                                 │
│  Buy the UltraBook Air 13.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: The client's original request was: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a          │
│  budget-conscious client who mostly does email, browsing, and light office work.'                               │
│                                                                                                                 │
│  Using ONLY the analyst's computed comparison, write a short client-ready recommendation. Cite the exact price  │
│  delta and at least one other computed number. End with exactly one explicit recommended laptop -- no 'it       │
│  depends' hedging.                                                                                              │
│  Agent: Client-Facing Recommendation Writer                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL RECOMMENDATION (sequential) ===

For your daily routine of email, web browsing, and light office work, paying an extra $600.00 for the UltraBook Pro 14 does not make financial sense. The UltraBook Air 13 easily handles these tasks while saving you a significant amount of money. Furthermore, its user rating of 4.3 out of 5 proves it is a reliable and well-liked machine for everyday use. 

Buy the UltraBook Air 13.


In [9]:
sequential_usage = sequential_crew.usage_metrics
print("Sequential run token usage:")
print(sequential_usage)

Sequential run token usage:
total_tokens=13608 prompt_tokens=12303 cached_prompt_tokens=0 completion_tokens=1305 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=21


### Sample run (captured from the cell above)

This is the real log captured from running the sequential crew, not a mock example.

```
[Laptop Product Researcher] Final Answer:
Name: UltraBook Air 13
Price: $899
RAM: 8GB
Storage: 256GB
Rating: 4.3

Name: UltraBook Pro 14
Price: $1499
RAM: 16GB
Storage: 512GB
Rating: 4.6

[Pricing & Value Analyst] Final Answer:
Price delta: $600.0
UltraBook Air 13 price per GB of RAM: $112.375
UltraBook Pro 14 price per GB of RAM: $93.6875
Trade off: For the $600.0 extra cost, the UltraBook Pro 14 delivers double the RAM (16GB vs 8GB), double the storage (512GB vs 256GB), and a slightly higher rating (4.6 vs 4.3), which works out to a lower cost per GB of RAM despite the higher total price.

[Client Facing Recommendation Writer] Final Answer:
For your daily routine of email, web browsing, and light office work, paying an extra $600.00 for the UltraBook Pro 14 does not make financial sense. The UltraBook Air 13 easily handles these tasks while saving you a significant amount of money. Furthermore, its user rating of 4.3 out of 5 proves it is a reliable and well liked machine for everyday use.

Buy the UltraBook Air 13.
```

### Where the format work paid off

In this captured run the researcher already returned the exact key value block on the first attempt, so nothing broke here. That is the direct result of a fix made earlier while iterating on the prompts. The first draft of `research_task.expected_output` simply said "report the specs for both laptops." Left open like that, the researcher sometimes answered in a conversational paragraph such as "The Air 13 comes in at $899 with 8GB of RAM, while the Pro 14 runs $1499 with 16GB..." which reads fine to a person but is not something the analyst can parse the same way every time: the price sometimes appeared before the product name, sometimes after, and RAM was occasionally written as "8 gigs" instead of a bare number. That inconsistency showed up downstream as the analyst occasionally computing the delta in the wrong direction. The fix was to tighten `expected_output` to demand an exact `key: value` block per product (shown in the task definitions above) and to add an explicit line to the researcher's task description telling it not to compute or summarize, only report what the tool returns. Pinning the shape of the hand off, not just its content, is what made the analyst's parsing reliable, and this run confirms the fix held.

## Task 4 — Hierarchical delegation

Same three specialists, but now a **manager agent** sits above them,
decides which sub-agent should handle each piece of work, and can send
work back if it isn't good enough. In `Process.hierarchical` you don't
pin `agent=` on each task up front -- the manager assigns them at
runtime -- and the manager needs its own `llm` (`manager_llm`) or an
explicit `manager_agent`. We use an explicit manager agent so its
reviewing behavior is visible in the log.

> **Free-tier note:** `gemini-3.5-flash-lite`'s free tier caps requests at
> 15/minute. Hierarchical delegation adds the manager's own reasoning and
> `delegate_work_to_coworker` calls on top of the 3 worker calls, so it can
> burn through that quota noticeably faster than the sequential run does
> -- an empirical version of the "higher token/call cost" point made in
> the comparison below. `AGENT_MAX_RPM` (set on every agent and both
> crews) keeps CrewAI's own request rate under the provider limit, and
> the `time.sleep(20)` before the hierarchical run gives the quota a
> moment to recover from the sequential run. If you still hit a 429 on a
> slower connection or a busier account, just re-run the cell -- CrewAI
> reports the provider's suggested retry delay in the error, and litellm
> retries automatically on most transient rate-limit errors.

In [10]:
manager = Agent(
    role="Recommendation Delivery Manager",
    goal=(
        "Deliver one accurate, client-ready laptop recommendation by "
        "delegating research, analysis, and writing to the right "
        "specialist, and rejecting any sub-agent output that skips a "
        "required number or invents a fact instead of sourcing it."
    ),
    backstory=(
        "A delivery lead who has shipped client recommendations for years "
        "and has been burned before by a writer who 'rounded' a number the "
        "analyst never actually computed. Reviews every hand-off before "
        "it moves to the next stage."
    ),
    llm=manager_llm,
    allow_delegation=True,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

# Hierarchical tasks: no `agent=` pinned -- the manager assigns these to
# researcher / analyst / writer at runtime. Descriptions/expected_output
# stay identical to the sequential version so the comparison is fair.
h_research_task = Task(
    description=research_task.description,
    expected_output=research_task.expected_output,
)
h_analysis_task = Task(
    description=analysis_task.description,
    expected_output=analysis_task.expected_output,
    context=[h_research_task],
)
h_writing_task = Task(
    description=writing_task.description,
    expected_output=writing_task.expected_output,
    context=[h_research_task, h_analysis_task],
)

hierarchical_crew = Crew(
    agents=[researcher, analyst, writer],   # manager_agent is separate, not in this list
    tasks=[h_research_task, h_analysis_task, h_writing_task],
    process=Process.hierarchical,
    manager_agent=manager,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

# Hierarchical delegation makes noticeably more LLM calls than sequential
# (manager reasoning + delegate_work_to_coworker calls on top of the 3
# worker calls) -- pause briefly first so this run starts with fresh
# per-minute quota instead of picking up wherever the sequential run left
# off. Only matters on the free tier; harmless otherwise.
import time
time.sleep(20)

hierarchical_result = await hierarchical_crew.kickoff_async()
print("\n=== FINAL RECOMMENDATION (hierarchical) ===\n")
print(hierarchical_result.raw)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 84cc1534-1af0-4ea6-90e5-60838acc86b6                                                                       │
│  Final Output: For your daily routine of email, web browsing, and light office work, paying an extra $600.00    │
│  for the UltraBook Pro 14 does not make financial sense. The UltraBook Air 13 easily handles these tasks while  │
│  saving you a significant amount of money. Furthermore, its user rating of 4.3 out of 5 proves it is a          │
│  reliable and well-liked machine for everyday use.                                                              │
│                                                                                                                 │
│  Buy the UltraBook Air 13.                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a7ee059f-841c-4a62-9685-537217f4606c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The client's request is: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a budget-conscious   │
│  client who mostly does email, browsing, and light office work.'                                                │
│                                                                                                                 │
│  Use the lookup_product_price tool to retrieve the full spec sheet for BOTH laptops named in the request. Do    │
│  not compute anything and do not recommend anything -- only report what the tool returns.                       │
│  ID: 522950e2-e673-4a03-b537-b397be5ad229                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Recommendation Delivery Manager                                                                         │
│                                                                                                                 │
│  Task: The client's request is: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a budget-conscious   │
│  client who mostly does email, browsing, and light office work.'                                                │
│                                                                                                                 │
│  Use the lookup_product_price tool to retrieve the full spec sheet for BOTH laptops named in the request. Do    │
│  not compute anything and do not recommend anything -- only report what the tool returns.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'Laptop Product Researcher', 'task': 'Retrieve the full spec sheets/pricing for UltraBook   │
│  Air 13 and UltraBook Pro 14 using the lookup_product_price tool.', 'context': 'The client wants t...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Recommendation Delivery Manager                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  For your daily routine of email, web browsing, and light office tasks, you do not need to overspend on         │
│  heavy-duty hardware. While the UltraBook Pro 14 offers a better efficiency of $93.69 per GB of RAM, it comes   │
│  with a steep $600.00 price delta that your budget simply does not require for basic use. The UltraBook Air 13  │
│  handles your exact workload comfortably at $899.00 without making you pay for extra power you will never use.  │
│                                                                                                                 │
│  Buy the UltraBook Air 13.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: The client's original request was: 'Compare the UltraBook Air 13 and the UltraBook Pro 14 for a          │
│  budget-conscious client who mostly does email, browsing, and light office work.'                               │
│                                                                                                                 │
│  Using ONLY the analyst's computed comparison, write a short client-ready recommendation. Cite the exact price  │
│  delta and at least one other computed number. End with exactly one explicit recommended laptop -- no 'it       │
│  depends' hedging.                                                                                              │
│  Agent: Recommendation Delivery Manager                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL RECOMMENDATION (hierarchical) ===

For your daily routine of email, web browsing, and light office tasks, you do not need to overspend on heavy-duty hardware. While the UltraBook Pro 14 offers a better efficiency of $93.69 per GB of RAM, it comes with a steep $600.00 price delta that your budget simply does not require for basic use. The UltraBook Air 13 handles your exact workload comfortably at $899.00 without making you pay for extra power you will never use. 

Buy the UltraBook Air 13.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [11]:
hierarchical_usage = hierarchical_crew.usage_metrics
print("Hierarchical run token usage:")
print(hierarchical_usage)

Hierarchical run token usage:
total_tokens=36270 prompt_tokens=31827 cached_prompt_tokens=0 completion_tokens=4443 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=51


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a7ee059f-841c-4a62-9685-537217f4606c                                                                       │
│  Final Output: For your daily routine of email, web browsing, and light office tasks, you do not need to        │
│  overspend on heavy-duty hardware. While the UltraBook Pro 14 offers a better efficiency of $93.69 per GB of    │
│  RAM, it comes with a steep $600.00 price delta that your budget simply does not require for basic use. The     │
│  UltraBook Air 13 handles your exact workload comfortably at $899.00 without making you pay for extra power     │
│  you will never use.                                                                                            │
│                                                                                                                 │
│  Buy the UltraBook Air 13.                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Sequential and hierarchical comparison

For this task, a fixed three stage pipeline with no real ambiguity about who should do what, the two processes converged on essentially the same final recommendation in the runs captured above, because there was no genuine delegation decision for the manager to add value on. The differences showed up in overhead, not quality:

| | Quality | Latency and token cost | Reliability |
|---|---|---|---|
| **Sequential** | Same final answer quality on this task. The pipeline order is fixed and unambiguous, so there is nothing for a manager to improve. | Lower: 21 successful calls and 13,608 tokens in the captured run, one round of calls per agent's task, no extra delegation round trips. | High: deterministic order every run; the only failure mode is a single agent's own output. |
| **Hierarchical** | Same quality in the captured run. The manager did not need to catch or redo anything here, since every sub agent's first attempt was already correct. | Higher: 51 successful calls and 36,270 tokens in the captured run, about 2.4 times the calls and 2.7 times the tokens of sequential, from the manager's own reasoning plus its delegate_work_to_coworker calls on top of the 3 worker calls. | Slightly lower on paper, since there is one more agent whose output or delegation choice can go wrong, but the review step can also catch a bad worker output that sequential would have shipped as is. |

| | Pros | Cons | When to use |
|---|---|---|---|
| **Sequential** | Cheap, fast, fully deterministic order, easy to debug from a single linear log | No error recovery if one stage's output is subtly wrong; it just flows downstream | The task's stage order is fixed and known ahead of time, like this one |
| **Hierarchical** | Manager can catch bad handoffs, redelegate, and handle tasks where the right agent is not obvious upfront | More tokens, more latency, one more point of failure in the manager itself | The task has real delegation ambiguity, needs quality review between stages, or the set of required steps is not fixed in advance |

## Task 5 — Evaluation & cost awareness

### Token usage & approximate cost

In [12]:
# Adjust these to the current published Gemini Flash-Lite rate before
# trusting the dollar figures -- pricing changes over time and this
# notebook doesn't fetch it live.
COST_PER_1M_INPUT_TOKENS_USD = 0.10
COST_PER_1M_OUTPUT_TOKENS_USD = 0.40

def approx_cost(usage) -> float:
    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    return (
        prompt_tokens / 1_000_000 * COST_PER_1M_INPUT_TOKENS_USD
        + completion_tokens / 1_000_000 * COST_PER_1M_OUTPUT_TOKENS_USD
    )

def summarize(label, usage):
    total = getattr(usage, "total_tokens", None)
    prompt = getattr(usage, "prompt_tokens", None)
    completion = getattr(usage, "completion_tokens", None)
    requests = getattr(usage, "successful_requests", None)
    print(f"{label}:")
    print(f"  total tokens:      {total}")
    print(f"  prompt tokens:     {prompt}")
    print(f"  completion tokens: {completion}")
    print(f"  successful calls:  {requests}")
    print(f"  approx cost:       ${approx_cost(usage):.5f}")
    print()

summarize("Sequential crew", sequential_usage)
summarize("Hierarchical crew", hierarchical_usage)

print(
    "Day 3 single-agent LangGraph reference (from that notebook's own run): "
    "1 model per generate + 1 per critique per pass (2 calls on a clean "
    "pass, up to 2 + 2*retries under revision) -- fewer total LLM calls "
    "than either crew above for this size of task, since there's no "
    "separate researcher/analyst/writer/manager hand-off overhead."
)

Sequential crew:
  total tokens:      13608
  prompt tokens:     12303
  completion tokens: 1305
  successful calls:  21
  approx cost:       $0.00175

Hierarchical crew:
  total tokens:      36270
  prompt tokens:     31827
  completion tokens: 4443
  successful calls:  51
  approx cost:       $0.00496

Day 3 single-agent LangGraph reference (from that notebook's own run): 1 model per generate + 1 per critique per pass (2 calls on a clean pass, up to 2 + 2*retries under revision) -- fewer total LLM calls than either crew above for this size of task, since there's no separate researcher/analyst/writer/manager hand-off overhead.


### Success criteria (3 simple, task appropriate checks)

1. **Factual grounding**: every number in the final recommendation (price, delta, RAM, and so on) traces back to an actual tool call output, with nothing invented or rounded beyond what the tool returned.
2. **Completeness**: the final answer names both candidate laptops, states the price delta, and ends with exactly one explicit recommended product (not "it depends").
3. **Tone**: plain, client appropriate language with no leaked internal reasoning, tool call syntax, or agent role play ("As the analyst, I...") in the final text.

### Manual scoring of the captured runs (1 to 5 scale per criterion)

| Run | Grounding | Completeness | Tone | Notes |
|---|---|---|---|---|
| Sequential (this notebook's run) | 5 | 5 | 5 | Every number, the $600 delta and the 4.3 rating, traces straight back to a tool call. Both laptops are named, the delta is stated, and it closes with one explicit pick: "Buy the UltraBook Air 13." No jargon, no leaked reasoning. |
| Hierarchical (this notebook's run) | 5 | 5 | 5 | Same grounding quality, and it goes further by citing the $93.69 per GB figure the analyst computed. Both laptops named, delta stated, one clear pick, clean client facing tone throughout. |

Only two real runs were captured here, one per process, since each run consumes real API quota. That is fewer than the three runs a fuller evaluation would ideally use. For a third data point, rerun either crew once more with `await sequential_crew.kickoff_async()` or `await hierarchical_crew.kickoff_async()` and score it the same way; the `verbose=True` logs already show everything needed to check grounding and tone by hand.

### Was the crew worth it for this task?

Based on the two real runs captured above, both processes produced an equally well grounded, equally complete, equally well toned recommendation, so adding the manager bought no quality gain here. What differed was pure overhead: the hierarchical run used 36,270 tokens across 51 successful calls versus the sequential run's 13,608 tokens across 21 calls, roughly 2.7 times the tokens and 2.4 times the calls for a result judged identical on all three criteria. Day 3's single LangGraph agent solves the same underlying problem in even fewer calls than the sequential crew, with a simpler, single log debugging story. For a task this small, four laptops, one price delta, one paragraph of client facing prose, a multi agent crew is not worth its added cost, and hierarchical delegation is worth it even less. That balance would likely shift once the task has genuinely separable work at a larger scale, more products to research at once, or a real need for a manager to catch a bad handoff before a client sees it, none of which this four laptop comparison actually has.

## Final comparison: sequential vs hierarchical vs single agent (Day 3)

| | Day 3 StateGraph (single agent) | CrewAI sequential | CrewAI hierarchical |
|---|---|---|---|
| Agents | 1 (does plan, retrieve, generate, and critique itself) | 3 specialists, fixed order | 3 specialists plus 1 manager |
| Control flow | Explicit graph with a real self correction cycle (`critique` can loop back to `generate`) | Fixed linear pipeline, no loop back | Manager can re delegate, closest thing to a loop back |
| LLM calls (this run) | 2 (generate plus critique) on a clean pass | 21 successful calls, measured | 51 successful calls, measured, about 2.4 times sequential |
| Token cost (this run) | Lowest, no comparable run captured here | 13,608 tokens, about $0.00175 | 36,270 tokens, about $0.00496, about 2.7 times sequential |
| Best fit | Tasks needing bounded retries, pauses for a person to step in, or state that can be saved and resumed | Tasks with a known, fixed sequence of specialized steps | Tasks with real delegation ambiguity or a need for review between stages |

**Bottom line:** all three can solve this particular laptop recommendation task, and on the runs captured above sequential and hierarchical produced equally good answers, so the extra cost of hierarchical delegation did not buy extra quality here. They still trade off differently as the task's shape changes: explicit self correction and the ability to pause favor Day 3's graph, a fixed specialist pipeline favors CrewAI sequential, and genuine delegation or review needs favor CrewAI hierarchical, when the task actually has that kind of ambiguity.